# DR-TMLE: an interval when the recorded assignment rule is hard to model

This notebook fits DR-TMLE, a TMLE variant that solves two extra score equations. The extra
equations protect the interval when one nuisance model converges to the wrong limit. Each step
shows its code, its output, and what the output tells you. The
[DR-TMLE reference](../technical-reference/dr-tmle/index.md) holds the theorem, the refusals, and
the release claim. If TMLE is new to you, start with
[point-treatment TMLE](point-treatment-tmle.ipynb).

## The applied question

The navigation program repeats its evaluation of the offer with 2,000 discharges. The program
logged every common cause of assignment and outcome, and a flexible model can learn the outcome.
The recorded assignment rule has thresholds and interactions. A main-effects logistic model of that
rule is therefore misspecified.

The program analyst asks one question. Can the program report an interval for the average treatment
effect (ATE) when the assignment model is the doubtful one? An unrecorded common cause is a
different failure, and DR-TMLE does not repair it.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| say which nuisance failure DR-TMLE addresses, and which it does not | "Why this method" |
| check that DR-TMLE is available for your estimand | Step 4 |
| configure the reduced regressions and the guard | Step 5 |
| confirm that an empty guard reproduces the ordinary TMLE | Step 6 |
| explain why solved score equations do not certify the fits | Step 7 |
| read the correction report and the reduced-regression diagnostics | Step 8 |


## Why this method

An ordinary TMLE stays consistent when one nuisance is consistent. Its interval needs both
nuisances to converge fast enough. The table compares the two estimators for the case this page
is about.

| estimator | when the assignment model converges to the wrong limit |
| --- | --- |
| ordinary TMLE | stays consistent. If the outcome fit converges too slowly, the remainder can dominate the root-n scale. The usual influence-curve interval then need not attain nominal coverage |
| DR-TMLE | solves two extra score equations built from reduced-dimension regressions. Under Theorem 1 of Benkeser et al. (2017), it stays asymptotically linear, given [rate conditions](../technical-reference/dr-tmle/theorem.md#the-remainder-terms-and-the-rate-conditions) on the outcome fit and the reduced regressions |

When both nuisances are consistent, the corrections converge to zero. DR-TMLE then has no
asymptotic advantage and adds finite-sample cost.
[What this solves](../technical-reference/dr-tmle/index.md#what-this-solves) gives the remainder
argument.

| term | plain meaning |
| --- | --- |
| estimand | the number the question asks for, written before any model is chosen. See [estimands](../user-guide/estimands.md) |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g. See [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| double robustness | the point estimate stays consistent when either nuisance model is consistent. See [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| targeting | a small update to Q, weighted by g, that removes first-order bias. See [targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) |
| influence curve | how much each row moves the estimate. Its variance gives the standard error. See [inference](../technical-reference/inference.md) |
| cross-fitting | each row's nuisance prediction comes from models fit without that row. See [CV-TMLE](../technical-reference/cv-tmle.md) |
| reduced regression | a regression with one input, the fitted value of the other nuisance. DR-TMLE fits three. See [the algorithm](../technical-reference/dr-tmle/index.md#the-algorithm-as-implemented) |
| score equation | a mean over the rows that targeting drives to zero. See [DR-TMLE diagnostics](../technical-reference/dr-tmle/diagnostics.md) |


## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
from dataclasses import replace

import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The data come from `make_nonlinear_ate`, the synthetic law of the point-treatment tutorial, at a
new size and seed. The code renames the generator's columns to the program's names. It prints the
first rows, the share of discharges offered navigation, and the true values of the law.


In [2]:
from cleverly.datasets import make_nonlinear_ate

frame, truth = make_nonlinear_ate(n=2_000, seed=55)
frame = frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "discharge_risk",
        "W2": "prior_utilization",
        "W3": "medication_burden",
        "W4": "age",
    }
)
print("rows and columns:", frame.shape)
print(frame.head().round(3))
print()
print("share offered navigation:", round(float(frame["transition_navigation"].mean()), 3))
print("known values of the synthetic law:")
for key in ("ey1", "ey0", "ate"):
    print(f"  {key}: {truth[key]:.3f}")

rows and columns: (2000, 6)
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             8.092                    0.0           0.842             -2.976             -0.305  1.450
1             3.733                    1.0          -1.244              0.053              1.500 -1.168
2             4.745                    0.0           0.811              1.899              0.447  1.629
3             4.675                    1.0          -0.138             -0.421              0.466 -1.440
4             4.122                    1.0           1.110              0.045             -1.293  1.132

share offered navigation: 0.45
known values of the synthetic law:
  ey1: 3.669
  ey0: 1.919
  ate: 1.750


**What this output tells you.** The frame has 2,000 discharges and six columns. The share offered
navigation is 0.45. The true arm means are 3.669 with the offer and 1.919 without it, so the true
ATE is 1.750.

| feature of the law | what it means for this page |
| --- | --- |
| the true propensity has a squared term, an interaction, and a threshold | a main-effects logistic model of assignment is misspecified. It converges to the wrong limit |
| the true outcome mean is nonlinear | gradient boosting can learn it, so the outcome fit is the credible nuisance |
| the four baseline covariates are standardized (mean 0, SD 1) | a negative value is below the average |

The generator `nonlinear_dgp` in `cleverly.datasets` defines both functions. A real program has no
`truth`, and a real analyst does not know which nuisance is misspecified. Every comparison against
the truth below is a teaching device.


## Step 3: write the protocol

A `StudyProtocol` records the scientific design before any model runs. Every result fitted from it
carries its fingerprint. This page keeps the
[shared study design](index.md#the-shared-study-design), and
[point-treatment TMLE](point-treatment-tmle.ipynb) explains each field.


In [3]:
from cleverly import StudyProtocol

protocol = StudyProtocol(
    target_population=(
        "Adults with a discharge-home order at a participating hospital "
        "during the enrollment period"
    ),
    eligibility=(
        "Age 18 years or older",
        "Discharge home ordered at a participating hospital",
    ),
    time_zero="Discharge-home order, after baseline measurement and before the navigation offer",
    treatment_strategies=(
        "Offer standard transition navigation",
        "Provide usual discharge support",
    ),
    treatment_versions=(
        "Bedside transition plan and two scheduled navigator contacts within 30 days",
        "No access to the transition-navigation offer",
    ),
    outcome="Patient-reported transition score",
    horizon="30 days after discharge",
    intercurrent_event_handling=(
        "Use the transition score regardless of readmission",
        "Analyze the offer regardless of completed contacts",
        "The protocol scores death before day 30 as the worst transition score "
        "(composite strategy)",
    ),
    interference_unit="Individual patient",
    assumption_rationale=(
        "The program logged every input of the recorded assignment rule, "
        "and the baseline variables cover the measured common causes",
        "The standardized offer and version records support consistency",
        "Reserved navigator capacity and access controls support no interference",
    ),
)
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 8f39186216794c1f
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['The program logged every input of the 

**What this output tells you.** The first line gives the schema version and the fingerprint
`8f39186216794c1f`. The fields match the point-treatment protocol, except the first assumption rationale.
That entry records that the program logged every input of the recorded assignment rule.

No protocol field records which nuisance model the analyst doubts. That doubt is an analysis
choice, not a design element. The `DRTMLEMethod` in Step 5 owns it through `guard=`, and the typed
estimand in Step 4 owns the contrast.


## Step 4: design and identification

DR-TMLE targets the same parameter as the ordinary TMLE, under the same assumptions. The design
keeps every common cause in the adjustment set. The code identifies the ATE and prints the method
catalog for it. It then asks the catalog about the average treatment effect on the treated (ATT).


In [4]:
from cleverly import ATE, ATT, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "prior_utilization", "medication_burden", "age"),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))
print(effect.summary())

print()
print("methods for the ATE:")
for method in effect.available_methods():
    print(f"  {method.name}: {method.available}")
att_catalog = {
    method.name: method for method in study.identify(ATT(reference=0)).available_methods()
}
print("drtmle for the ATT:", att_catalog["drtmle"].available, "-", att_catalog["drtmle"].reason)

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 8f39186216794c1f
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measu

**What this output tells you.** The summary gives the observed-data formula, the two required
nuisances, and four assumptions. It then repeats the stored protocol. The catalog lists `drtmle`
as available for this ATE. For the ATT it prints `False` and the reason.

| catalog outcome | when it happens |
| --- | --- |
| available | a point-treatment arm contrast, such as this ATE |
| unavailable in the catalog | an `ATT`, `ATC`, MSM, or intervention-axis effect. Selecting `drtmle` raises before any nuisance is fitted |
| refused at fit time | the other refusals, such as composition with collaborative TMLE. [Refused by name](../technical-reference/dr-tmle/supported-estimands.md#refused-by-name) lists them |

DR-TMLE relaxes none of the four assumptions. It changes only the nuisance conditions that the
interval needs.


## Step 5: estimate with DR-TMLE

The primary nuisances are the analyst's: a flexible outcome regression and a crude assignment
model. Each reduced regression has one input, so a spline can fit it fast. Each reduction below is
a Super Learner over a linear and a spline candidate. The
[`drtmle` vignette](https://github.com/benkeser/drtmle/blob/538a3a264c1ca984b6d88978ca7f96165f43152c/vignettes/using_drtmle.Rmd)
uses the same pairing, `SL.glm` and `SL.gam`.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | main-effects logistic regression | fits g. On this law it is misspecified by construction |
| `CrossFitting(n_folds=3)` | three folds | predicts each row from models that did not see that row. The default reduced cross-fitting refuses fewer than three folds |
| `reduced_outcome_learner` | Super Learner, linear and spline | fits the conditional means $Q_r$ and $g_{r,2}$ |
| `reduced_treatment_learner` | Super Learner, logistic and spline | fits the probability $g_{r,1}$ |
| `guard` | the default, `("Q", "g")` | poses both extra equations. `guard=("g",)` guards only against the assignment model and reports $D = D^{*} - D^{*}_{Q}$ |
| `Runtime(random_state=55, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |


In [5]:
from cleverly import CrossFitting, DRTMLEMethod, ModelSpec, Runtime, SuperLearner

models = ModelSpec(
    outcome_learner=HistGradientBoostingRegressor(random_state=55),
    treatment_learner=LogisticRegression(max_iter=1000),
)
folds = CrossFitting(n_folds=3)
runtime = Runtime(random_state=55, n_jobs=1)

reduced_outcome = SuperLearner(
    library=[
        ("linear", LinearRegression()),
        (
            "spline",
            make_pipeline(SplineTransformer(n_knots=5, knots="quantile"), LinearRegression()),
        ),
    ],
    task="regression",
    n_folds=3,
    random_state=55,
)
reduced_treatment = SuperLearner(
    library=[
        ("logistic", LogisticRegression()),
        (
            "spline",
            make_pipeline(
                SplineTransformer(n_knots=5, knots="quantile"), LogisticRegression(max_iter=1000)
            ),
        ),
    ],
    task="classification",
    n_folds=3,
    random_state=55,
)
drtmle = DRTMLEMethod(
    models=models,
    cross_fitting=folds,
    runtime=runtime,
    reduced_outcome_learner=reduced_outcome,
    reduced_treatment_learner=reduced_treatment,
)
guarded = effect.estimate(method=drtmle)
print(guarded.summary())
print()
print("guard:", drtmle.guard, "| reduction:", drtmle.reduction)
point = guarded["ate"]
print(f"estimate:        {point.psi:.3f}")
print(f"standard error:  {point.std_error:.4f}")
print(f"95% CI:          ({point.ci[0]:.3f}, {point.ci[1]:.3f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 2000; covariates = 4; P(A=1) = 0.4495
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 8f39186216794c1f
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseli

**What this output tells you.** The summary names the construction as stacked CV-TMLE, with
nuisances cross-fitted over 3 folds. It shows the propensity truncation bound, [0.01471, 0.9853].
The line after the summary confirms the default guard and the univariate reduction.

The estimate is 1.707 with a standard error of 0.0565. The 95% interval (1.597, 1.818) contains
the true ATE of 1.750. That is one draw, not a coverage result.

The summary has the same layout as an ordinary TMLE summary. It does not print the reduced
regressions or the corrections. Steps 7 and 8 read them.


## Step 6: compare with the ordinary TMLE

An empty guard solves no extra equation, so it must reproduce the ordinary TMLE exactly. The code
fits the ordinary TMLE with the same learners, folds, and seed. It fits DR-TMLE with `guard=()`
and compares the two estimates. It then prints the ordinary fit beside the guarded fit.


In [6]:
from cleverly import TMLEMethod

ordinary = effect.estimate(method=TMLEMethod(models=models, cross_fitting=folds, runtime=runtime))
empty_guard = effect.estimate(method=replace(drtmle, guard=()))
print("empty guard reproduces the ordinary TMLE:", ordinary["ate"].psi == empty_guard["ate"].psi)
print()
for label, fitted in (("ordinary TMLE", ordinary), ("DR-TMLE", guarded)):
    point = fitted["ate"]
    low, high = point.ci
    print(f"{label:14s} psi={point.psi:.3f}  se={point.std_error:.4f}  CI=({low:.3f}, {high:.3f})")
print(f"population ATE: {truth['ate']:.3f}")
shift = (guarded["ate"].psi - ordinary["ate"].psi) / ordinary["ate"].std_error
se_ratio = guarded["ate"].std_error / ordinary["ate"].std_error
print(f"DR-TMLE shift, in ordinary standard errors: {shift:+.2f}")
print(f"standard-error ratio, DR-TMLE / ordinary:   {se_ratio:.2f}")

empty guard reproduces the ordinary TMLE: True

ordinary TMLE  psi=1.728  se=0.0548  CI=(1.621, 1.835)
DR-TMLE        psi=1.707  se=0.0565  CI=(1.597, 1.818)
population ATE: 1.750
DR-TMLE shift, in ordinary standard errors: -0.38
standard-error ratio, DR-TMLE / ordinary:   1.03


**What this output tells you.** The first line prints `True`. The empty-guard estimate equals the
ordinary estimate exactly. The equality fixes what the variant is: the same estimator plus extra
equations, not a different target.

| quantity | ordinary TMLE | DR-TMLE |
| --- | --- | --- |
| estimate | 1.728 | 1.707 |
| standard error | 0.0548 | 0.0565 |
| 95% interval | (1.621, 1.835) | (1.597, 1.818) |

The guarded estimate moves because the extra fluctuations change the targeted fits. On this draw
it moves by -0.38 ordinary standard errors, less than one. The standard-error ratio is 1.03. That
resemblance says nothing about either interval's coverage.


## Step 7: the failure mode, solved scores do not certify the fits

Score equations describe the targeting step. They do not measure how far a fitted function is from
the true one. The code refits DR-TMLE with constant reductions. A constant reduction ignores its
input, so it cannot follow any dependence on the fitted nuisance. The code prints each fit's
estimate and its two score checks.


In [7]:
crude = replace(
    drtmle,
    reduced_outcome_learner=DummyRegressor(),
    reduced_treatment_learner=DummyClassifier(strategy="prior"),
)
crude_fit = effect.estimate(method=crude)
for label, fitted in (("spline reductions", guarded), ("constant reductions", crude_fit)):
    point = fitted["ate"]
    scores = fitted.diagnostics.score_equations()
    corrections = fitted.diagnostics.corrections()
    print(
        f"{label:20s} psi={point.psi:.3f}  se={point.std_error:.4f}  "
        f"score equations passed={scores.passed}  corrections passed={corrections.passed}"
    )

spline reductions    psi=1.707  se=0.0565  score equations passed=True  corrections passed=True
constant reductions  psi=1.724  se=0.0548  score equations passed=True  corrections passed=True


**What this output tells you.** Both fits print `passed=True` for the score equations and for the
corrections. The spline fit estimates 1.707. The constant fit estimates 1.724, with a standard
error of 0.0548. The two fits report different estimates, and the checks cannot tell them apart.

On this draw the constant fit lands nearer the true ATE of 1.750. That does not make it the better
fit. One draw cannot rank two reductions, and a real analysis has no truth to compare against.

[Solved scores do not establish nuisance consistency](../technical-reference/dr-tmle/diagnostics.md#solved-scores-do-not-establish-nuisance-consistency)
gives the exact-law test behind this rule. In that test, wrong reductions move the estimate, and
every score equation still passes.


## Step 8: diagnostics, what the fit can show

The combined assessment presents validation, diagnostics, and sensitivity together. The code
prints its summary and three retained outputs. They are the correction check, the nuisance report,
and the reduced-regression diagnostics. `guarded.extra["drtmle"].diagnostics` holds one Super
Learner record per arm and fold for each reduced regression.


In [8]:
assessment = guarded.assess()
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))
print()
print(assessment.report("corrections").summary())
print()
print(assessment.report("nuisance_models").summary())
print()
reduced = guarded.extra["drtmle"].diagnostics
for family, fits in reduced.items():
    print(family, "best candidate per arm and fold:", [fit.best for fit in fits])

Returned results
----------------
surface      operation            result                                                                                                          
-----------  -------------------  ----------------------------------------------------------------------------------------------------------------
validation   support              maximum truncated fraction 0.0%; minimum effective-sample-size ratio 90.6%                                      
validation   nuisance_models      2 nuisance model report(s) are available                                                                        
sensitivity  omitted_confounding  at the default strengths; cf_y=0.03, cf_d=0.03, rho=1; bias-adjusted interval [1.635, 1.779]                    
sensitivity  robustness_value     point robustness value 0.5072; confidence-limit value 0.486                                                     
sensitivity  elements             sigma2=1.286, nu2=4.34, max_bias=2.363            

**What this output tells you.** Read the four parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | `score_equations` and `corrections` pass, and no row needs attention |
| correction check | each extra equation per arm and its solved score. The contract is `theorem`, because no truncation is active |
| nuisance model diagnostics | `nuisance fits look reasonable`. The misspecified logistic propensity has a calibration slope of 0.9744 |
| reduced-regression diagnostics | the spline candidate has the lowest cross-validated risk in all six $g_{r,1}$ fits. The $Q_r$ and $g_{r,2}$ fits split between the two candidates |

The nuisance report looks reasonable for an assignment model that the analyst knows has the wrong
form. Held-out risk compares candidates, but it cannot measure distance from the true function.
The reduced-regression table is the one place the fit shows a choice that the theorem's conditions
depend on. In the $g_{r,1}$ fits, the data favor a nonlinear reduction.

The `sensitivity` rows address exchangeability, which DR-TMLE does not relax. Step 9 reads them.


## Step 9: sensitivity, what DR-TMLE does not relax

DR-TMLE protects the interval against one badly fitted nuisance. It does not protect against an
unmeasured confounder.
[Sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) asks how strong
such a confounder would need to be to change the conclusion. The assessment already holds the
robustness value and the omitted-confounding bounds, and the code prints both.


In [9]:
robustness = assessment.report("robustness_value")
bounds = assessment.report("omitted_confounding")
print(f"robustness value: {robustness['rv']:.3f} (confidence-limit value {robustness['rva']:.3f})")
print(bounds)

robustness value: 0.507 (confidence-limit value 0.486)
Omitted-variable sensitivity for 'ate'
--------------------------------------------
estimate 1.7072; maximal bias sqrt(sigma^2 nu^2) = 2.3629
assumed confounding: cf_y = 0.03, cf_d = 0.03, rho = 1 -> bias <= 0.071974
bias-adjusted bounds:  [1.6352, 1.7792]
with 95% one-sided CIs: [1.5424, 1.8722] (the sign of the effect survives)

robustness value RV   = 0.5072: a confounder explaining 50.7% of the residual variation in BOTH the outcome and treatment would move the estimate to 0.
robustness value RVa  = 0.4860: the same, for the 95% confidence bound.


**What this output tells you.** The robustness value is 0.507, and the confidence-limit value is
0.486. A confounder that explains 50.7% of the residual variation in both the outcome and the
treatment would move the estimate to zero. The value assumes worst-case alignment, `rho = 1`.

The bounds use the default strengths, `cf_y = 0.03` and `cf_d = 0.03`. At those strengths the
bias-adjusted bounds are [1.6352, 1.7792]. The default strengths are not calibrated to this
program. The assessment defers the benchmark that calibrates them, because the benchmark refits
the nuisances. [Point-treatment TMLE](point-treatment-tmle.ipynb) runs that benchmark.


## How far to trust this

| layer | establishes | does not establish |
| --- | --- | --- |
| `guard=()` equality | the variant reduces exactly to the ordinary estimator | anything about the guarded fit |
| the constant-reduction refit | that passing score checks do not separate two reductions | which reduction is adequate |
| the score and correction reports | the targeting solved all three equations, with no active truncation | the rate conditions behind the interval |
| the nuisance and reduced-regression reports | which candidate fit best under cross-validated risk | that any fitted function is consistent |
| the omitted-confounding rows | how much hidden confounding would move the estimate | that no hidden confounder exists |

DR-TMLE ships under **conditional validity**. The registered
[canonical DR-TMLE study](../technical-reference/method-evidence/canonical-dr-tmle.md) uses a
binary complete-data law. No registered study covers this continuous law with flexible learners.
The [DR-TMLE evidence](../technical-reference/dr-tmle/validation-programme.md) shows where the
interval fell short of nominal coverage.


## Where to go next

If your question is which baseline variables belong in the assignment model, read
[collaborative TMLE](collaborative-tmle.ipynb). The two methods do not compose. A reduced regression
conditions on the fitted assignment mechanism as a covariate, and the C-TMLE mechanism is
deliberately not an estimate of the true one. DR-TMLE raises that refusal at fit time.

When outcomes are also missing, double robustness takes a different shape. Read
[survey non-response](survey-nonresponse.ipynb).

The [examples index](index.md#the-program) lists every tutorial in the program.
